# Example of Tracking-60k

In [1]:
import os
import sys
sys.path.append('../src')

import torch
import torch.utils.benchmark as benchmark
from pathlib import Path

from transformer import Transformer
from trainer import run_one_epoch, init_metrics
from utils import get_loss
from utils.get_data import get_data_loader, get_dataset
from tqdm import tqdm

torch.set_num_threads(10)

In [2]:
def get_event_id_sector_from_str(name: str) -> tuple[int, int]:
    """
    Returns:
        Event id, sector Id
    """
    number_s = name.split(".")[0][len("data") :]
    evtid_s, sectorid_s = number_s.split("_s")
    evtid = int(evtid_s)
    sectorid = int(sectorid_s)
    return evtid, sectorid


def get_embedding(data, model, device):
    with torch.no_grad():
        return {"H": model(data.x, data.coords, torch.zeros(size=(data.x.shape[0], )).long().to(device))}

In [3]:
device = 'cuda:0'
dataset_name = 'tracking-60k'
batch_size = 1
model_configs = {'block_size': 100, 'n_hashes': 3, 'num_regions': 150, 'num_heads': 8, 'h_dim': 24, 'n_layers': 4, 'num_w_per_dist': 10}
torch.cuda.set_device(device)

In [4]:
dataset_dir = Path('../data/') / dataset_name.split("-")[0]
dataset = get_dataset(dataset_name, dataset_dir)

In [5]:
all_point_clouds = os.listdir(dataset.raw_dir)

In [6]:
data_list = []
for data in tqdm(dataset):
    evtid = data['evtid'].item()
    for point_cloud in all_point_clouds:
        if point_cloud == f'data{evtid}_s0.pt':
            break
    pt_evtid, sector = get_event_id_sector_from_str(point_cloud)
    assert pt_evtid == evtid
    pt_data = torch.load(Path(dataset.raw_dir) / point_cloud)

    assert torch.equal(data.particle_id, pt_data.particle_id)
    assert torch.equal(data.pt, pt_data.pt)
    assert torch.equal(data.reconstructable, pt_data.reconstructable)
    assert torch.equal(data.layer, pt_data.layer)

    assert pt_data.eta.shape[0] == data.x.shape[0]
    assert pt_data.n_hits.shape[0] == data.x.shape[0]
    assert pt_data.n_layers_hit.shape[0] == data.x.shape[0]

    data.eta = pt_data.eta
    data.n_hits = pt_data.n_hits
    data.n_layers_hit = pt_data.n_layers_hit

    data_list.append(data)

100%|██████████| 50/50 [00:00<00:00, 120.59it/s]


In [7]:
dataset.data, dataset.slices = dataset.collate(data_list)

In [8]:
loaders = get_data_loader(dataset, dataset.idx_split, batch_size=batch_size)

In [9]:
model = Transformer(in_dim=dataset.x_dim, coords_dim=dataset.coords_dim, num_classes=dataset.num_classes, **model_configs).to(device)

In [10]:
checkpoint = torch.load("./ckpt/tracking-60k-model.pt", map_location="cpu")
model.load_state_dict(checkpoint, strict=True)
model = model.to(device)

criterion = get_loss('infonce', {'dist_metric': 'l2_rbf', 'tau': 0.05})
metrics = init_metrics(dataset_name)

In [11]:
with torch.no_grad():
    model.eval()
    test_res = run_one_epoch(model, None, criterion, loaders["test"], "test", 0, device, metrics, None)

print(f"Test accuracy@0.9: {test_res['accuracy@0.9']:.4f}")

[Epoch 0] test , loss: 0.5744, acc: 0.9208, prec: 0.3808, recall: 0.9758: 100%|██████████| 5/5 [00:05<00:00,  1.15s/it]

Test accuracy@0.9: 0.9208


In [14]:
model.eval()
for i_batch, data in enumerate(tqdm(loaders["train"])):
    data = data.to(device)
    out = get_embedding(data, model, device)
    break

  0%|          | 0/40 [00:00<?, ?it/s]


In [18]:
out['H'].shape

torch.Size([62310, 12])

In [29]:
from sklearn.cluster import DBSCAN
from gnn_tracking_prev.metrics.cluster_metrics import tracking_metrics


In [21]:
clustering = DBSCAN(eps=0.5, min_samples=1).fit(out['H'].cpu().numpy())

In [28]:
clustering.labels_.shape

(62310,)

In [27]:
data.particle_id.shape

torch.Size([62310])

In [ ]:
pts = data.pt
reconstructable = data.reconstructable

res = tracking_metrics(
    truth=data.particle_id.cpu().numpy(),
    predicted=clustering.labels_ + 100,
    pts=pts.cpu().numpy(),
    reconstructable=reconstructable.cpu().numpy(),
    pt_thlds=[0.9],
    predicted_count_thld=1,
)

In [ ]:
res

{0.9: {'n_particles': 1523,
  'n_cleaned_clusters': 1415,
  'perfect': 0.6408404464871963,
  'double_majority': 0.8956007879185818,
  'lhc': 0.8862190812720848,
  'fake_perfect': 0.2882468811556139,
  'fake_double_majority': 0.0334865397242285,
  'fake_lhc': 0.1137809187279152}}

: 

In [13]:
from gnn_tracking.postprocessing.dbscanscanner import DBSCANHyperParamScannerFixed, DBSCANHyperParamScanner
import numpy as np

ImportError: cannot import name 'DBSCANHyperParamScannerFixed' from 'gnn_tracking_prev.postprocessing.dbscanscanner' (/usr/scratch/smiao35/projects/HEPTv2/example/gnn_tracking_prev/postprocessing/dbscanscanner.py)

In [21]:
scanner = DBSCANHyperParamScannerFixed(
    trials = [{"eps": eps, "min_samples": min_s} for eps in np.linspace(0.4, 0.6, 50) for min_s in [1]]
)

# scanner = DBSCANHyperParamScanner(n_trials=10, keep_best=1)

In [23]:
model.eval()
for i_batch, data in enumerate(tqdm(loaders["train"])):
    data = data.to(device)
    out = get_embedding(data, model, device)
    scanner(data, out, i_batch=i_batch, progress=True)
    print(scanner.get_foms()["trk.double_majority_pt0.9"])
    print(scanner.get_foms()["best_dbscan_eps"], scanner.get_foms()["best_dbscan_min_samples"])

  2%|▎         | 1/40 [00:09<06:05,  9.38s/it]

0.9038587311968607
0.5224489795918368 1.0


  5%|▌         | 2/40 [00:19<06:10,  9.76s/it]

0.9048175614239835
0.4938775510204082 1.0


  8%|▊         | 3/40 [00:27<05:32,  8.98s/it]

...

0.9127989855348079
0.47346938775510206 1.0


 98%|█████████▊| 39/40 [05:53<00:09,  9.27s/it]

0.9122571312048555
0.47346938775510206 1.0


100%|██████████| 40/40 [06:00<00:00,  9.00s/it]

0.9131363252423306
0.47346938775510206 1.0


# Benchmark Inference Speed

In [8]:
# cuda 12.1
model = torch.compile(model)

In [9]:
torch.set_float32_matmul_precision('high')
for data in loaders["test"]:
    if data.x.shape[0] > 60000:
        data = data.to(device)
        break

model.eval()
with torch.no_grad():
    t1 = benchmark.Timer(
        stmt=f"model(data.x, data.coords, data.batch)", setup=f"from __main__ import model, data"
    )
    m = t1.blocked_autorange(min_run_time=5)
print(m)

model(data.x, data.coords, data.batch)
setup: from __main__ import model, data
  Median: 29.96 ms
  IQR:    0.07 ms (29.92 to 29.99)
  167 measurements, 1 runs per measurement, 1 thread
